# v0.8.6 failure debugging: step-by-step placement of a WRONG TPC

Walks the **full v0.8.6 chain** — external frontend, **4x-smaller amplitude
perceiver**, merged pool, rescue, P2+police (**V2 skipped** — superseded by
joint+PASS 2, measured on 126 events), JOINT mean-field solver, error matrix,
PASS 2 trading with light-nominated candidates, swap fix, force coverage — on
one event, drawing the watched TPC's predicted-vs-actual light and the
reconstructed 3D view at every placement that touches it.

**Subject selection:** cell 2 scans the newest dumps (v0.8.6 geometry) and
lists *moderate* failure TPCs — wrong-energy fraction **10–20%** — with each
wrong cluster's full assignment-evolution story. Pick any row, set `EV, TPC`
in cell 1, rerun.

Needs a GPU node (the chain runs live; ~2 min per stage after model load).


In [1]:
import sys, time
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

TESTING = "/global/cfs/cdirs/dune/users/yuxuan/NDLAr-full/CLMatching_v0.1_backboneOnly/testing"
sys.path.insert(0, TESTING)
import v040_walkthrough as W

# ---- v0.8.6 configuration ----
W.FRONTEND_REPO = "/global/cfs/cdirs/dune/users/yuxuan/NDLAr-full/clmatching-frontend"
W.LIGHT_CKPT = ("/global/cfs/cdirs/dune/users/yuxuan/NDLAr-full/NewMLSection/"
                "runs/ndfull_small4x/best_model_arch.pt")   # 4x-smaller perceiver
W.SKIP_V2 = True          # canonical v0.8.6; False replays the old V2 stage

# ---- choose the subject (default: 17.8%-wrong TPC — see the gallery in cell 2) ----
FTAG, EV, TPC = "0000001", 1, 16
DATA = ("/global/cfs/cdirs/dunepro/people/abooth/nd-production/output/MiniProdN5/"
        "run-ndlar-flow/MiniProdN5p1_NDComplex_FHC.flow.full.sanddrift/FLOW/0000000")
FP = f"{DATA}/MiniProdN5p1_NDComplex_FHC.flow.full.sanddrift.{FTAG}.FLOW.hdf5"

from ML_NDfull_perceiver import load_tpc_geometries, DEFAULT_TPC_YAML
from visualizationTools import plot_tpc_light_and_charge
import plottingTools as pt
geom_map = load_tpc_geometries(DEFAULT_TPC_YAML)
print(f"subject: {FTAG} ev{EV} TPC {TPC}  (v0.8.6: small perceiver, no V2)")


subject: 0000001 ev1 TPC 16  (v0.8.6: small perceiver, no V2)


In [13]:
# ---- failure gallery: MODERATE (10-20% wrong) non-shower TPCs + failure stories ----
# scans the newest dumps (v0.8.6 geometry preferred) — the good teaching examples:
# enough wrong energy to matter, not hopeless showers.
import glob, json as _json, numpy as _np
for _cand in ("v096EVO_out", "v085EVO_out", "v060dump_out"):
    if glob.glob(f"{TESTING}/{_cand}/dumps/*.npz"):
        DUMPS = f"{TESTING}/{_cand}/dumps"; break
print(f"dumps: {DUMPS}")
LO, HI = 0.10, 0.20          # wrong-fraction band to surface (edit freely)
rows = []
for fp in sorted(glob.glob(f"{DUMPS}/{FTAG}_ev*.npz")):
    d = _np.load(fp, allow_pickle=True)
    e = _np.clip(_np.asarray(d["e"], float), 0, None); tpc_ = _np.asarray(d["tpc"])
    tsf = _np.asarray(d["ts_final"], float); tru = _np.asarray(d["truth_t0"], float)
    ok = _np.isfinite(tru) & _np.isfinite(tsf) & (tsf >= 0)
    wrong = ok & (_np.abs(tsf - tru) > 10)
    for t in _np.unique(tpc_[wrong]):
        a = ok & (tpc_ == t); etot = float(e[a].sum())
        ew = float(e[wrong & (tpc_ == t)].sum())
        if 30 < etot <= 1400 and LO <= ew / etot <= HI:
            rows.append((ew, ew / etot, int(d["event"]), int(t), etot, fp))
    d.close()
rows.sort(reverse=True)
print(f"{'ev':>3} {'tpc':>4} {'frac':>6} {'E_wrong':>8} {'E_tpc':>8}   (band {LO:.0%}-{HI:.0%})")
for ew, fr, ev_, t_, etot, _fp in rows[:12]:
    print(f"{ev_:3d} {t_:4d} {fr:6.1%} {ew:8.1f} {etot:8.1f}")

# ---- failure story of the selected subject (from the t_evo evolution record) ----
sub = [r for r in rows if r[2] == EV and r[3] == TPC] or rows[:1]
if sub:
    _fp = sub[0][5]
    d = _np.load(_fp, allow_pickle=True)
    if "t_evo" in d.files:
        import evo_gallery as EG
        ev_d = {k: d[k] for k in ("labels", "ts_final", "truth_t0", "e", "tpc")}
        ev_d["t_evo"] = {s: {int(l): t for l, t in m.items()}
                         for s, m in _json.loads(str(d["t_evo"])).items()}
        lab_rows = _json.loads(str(d["lab_rows"]))
        t_sel = sub[0][3]
        print(f"\nfailure story, ev{sub[0][2]} TPC {t_sel} (per wrong cluster):")
        for r in lab_rows:
            if t_sel not in r["tpcs"] or r["e_wrong"] < 5:
                continue
            hist, story = EG.label_story(ev_d, r["label"], r["truth"])
            path = " -> ".join(f"{t:.0f}" for _, t in hist[:6])
            print(f"  label {r['label']:5d} {r['e_wrong']:7.1f} MeV wrong | {story}")
            print(f"        t path: {path}  (truth {r['truth']})")
    d.close()
print("\n-> set EV, TPC in cell 1 and rerun to walk any of these live")


dumps: /global/cfs/cdirs/dune/users/yuxuan/NDLAr-full/CLMatching_v0.1_backboneOnly/testing/v096EVO_out/dumps
 ev  tpc   frac  E_wrong    E_tpc   (band 10%-20%)
  1   16  17.8%    222.0   1245.9
  9   57  18.8%    171.9    915.7
  7   19  14.3%    163.2   1138.6
 12    1  14.5%    157.4   1082.7
  5   16  16.7%    136.6    816.7
  1   61  13.3%    131.6    989.2
  2   67  10.1%    125.1   1236.0
  3   24  12.9%    121.9    941.1
 10   43  14.5%    117.5    810.7
  7   18  13.6%    112.9    827.9
  6   16  10.1%    110.4   1091.5
 11   43  14.3%    105.3    738.7

failure story, ev1 TPC 16 (per wrong cluster):
  label    28    15.8 MeV wrong | correct from backbone +pool
        t path: 589 -> 589 -> 589 -> 589 -> 589 -> 589  (truth 588.166357803857)
  label   431    21.6 MeV wrong | WRONG from placement [joint] (never touched again)
        t path: 589 -> 589 -> 589 -> 589 -> 589  (truth 480.8729650103487)
  label   436    10.5 MeV wrong | WRONG from placement [joint] (never touched aga

In [14]:
# heavy setup: models + v0.8.1 config (boundary 30, predict >=1.5 MeV, veto-4,
# vertex merge, external frontend)
ctx = W.load_chain()


[load_chain] amplitude perceiver -> /global/cfs/cdirs/dune/users/yuxuan/NDLAr-full/NewMLSection/runs/ndfull_small4x/best_model_arch.pt
[load_chain] FRONT-END clustering -> external:clmatching-frontend:params_full.json
[load_chain] models + config ready in 0.3s (boundary 30.0 MeV, predict >= 1.5 MeV, veto-4, vertex merge ON)


## Stage 1 — clustering & ML prediction

One call produces: RANSAC track/shower clustering (+ vertex-track merge), the
perceiver light-amplitude prediction for every cluster >= 1.5 MeV (GPU), and the
support channels. It also runs the production track placement, which stages
2/3.1 below re-do with the stack2 machinery — we keep only its t0 medians as
references. The cluster table for the watched TPC defines every object we will
follow.

In [15]:
S = W.first_stage(ctx, FP, EV, TPC)

# ---- cluster definition for this TPC ----
import pandas as pd
rows, n_noise = W.cluster_table(S)
df = pd.DataFrame(rows).sort_values("e_tpc", ascending=False)
print(f"clusters with hits in TPC {TPC}: {len(df)} (+ {n_noise} noise hits with no cluster)")
display(df)

[first_stage] 52.1s | clusters 167 backbone labels | watched TPC 16: 3367 hits, 1246 MeV
clusters with hits in TPC 16: 50 (+ 105 noise hits with no cluster)


,label,type,backbone,n_hits_tpc,e_tpc,e_total,tpcs,has_template,vertex_merged,truth_t0
0,15,track,True,518,155.6,434.6,"[9, 14, 16, 18]",True,False,609.603194
3,27,track,True,468,133.9,420.8,"[14, 16, 18]",True,False,617.857878
5,30,track,True,343,129.7,252.9,"[14, 16]",True,False,588.124308
4,28,track,True,262,108.0,208.5,"[14, 16]",True,False,588.166358
6,38,track,True,175,95.8,365.1,"[16, 17, 24, 25, 34]",True,False,349.813732
38,446,cluster,False,232,95.4,95.4,[16],True,False,588.111018
2,19,track,True,238,92.1,540.7,"[10, 12, 14, 16]",True,False,111.009572
7,40,track,True,164,71.2,71.2,[16],True,False,588.214319
1,18,track,True,152,58.6,537.8,"[10, 12, 14, 16]",True,False,67.839099
47,456,cluster,False,133,45.7,45.7,[16],True,False,588.288750


In [21]:
# ---- display machinery: fires on every base-image change of the watched TPC ----
_prev = {"base": None, "n": 0}

def show_state(stage, note, base_view=None, force=False):
    base = np.asarray(base_view if base_view is not None else W.current_base(S)[TPC], np.float32)
    changed = _prev["base"] is None or not np.allclose(base, _prev["base"], atol=1.0)
    if not (changed or force):
        return
    _prev["n"] += 1
    print(f"\n{'='*100}")
    print(f"  frame {_prev['n']:>3} | [{stage}]  {note}")
    print(f"{'='*100}")
    if changed:
        _prev["base"] = base.copy()
    # 1) the usual 2D light + charge display
    plot_tpc_light_and_charge(ev_id=EV, TPCid=TPC, hits_full_ev=S["hits_full_ev"],
                              actual=S["actual"][TPC].astype(np.float32),
                              predicted=base, geom_map=geom_map, individualScaling=False)
    plt.show()
    # 2) interactive 3D reconstructed display (gray = not yet assigned)
    ts = W.current_ts(S)
    m = S["hit_tpc"] == TPC
    safe = np.where(np.isfinite(ts) & (ts >= 0), ts, -1.0)
    g = pt.group_hits_by_time(safe[m], time_window=10)
    pt.plot_3d_clusters_with_t0(
        x=np.asarray(S["fs"].event.x)[m], y=np.asarray(S["fs"].event.y)[m],
        z=np.asarray(S["fs"].event.z)[m], labels=g, t0s=safe[m],
        energies=S["e_pos"][m], title=f"[{stage}] {note[:70]}", save_path=None)

# ---- frame 0: FRONT-END CLUSTERING map (real label ids — the clustering decision,
# nothing from truth) so "cluster i placed ..." in later frames is locatable.
import plotly.graph_objects as go

np.asarray(S["fs"].track_stage.base_image)[:] = 0.0     # reset walkthrough state
S["hit_t0"][:] = np.nan

MIN_E_LABEL, MAX_LEGEND = 5.0, 40      # color clusters >= 5 MeV (in this TPC), top 40 by energy
m = S["hit_tpc"] == TPC
labs = np.asarray(S["labels_global"])[m].astype(int)
xs, ys, zs = (np.asarray(S["fs"].event.x)[m], np.asarray(S["fs"].event.y)[m],
              np.asarray(S["fs"].event.z)[m])
e_m = S["e_pos"][m]

ids = [(float(e_m[labs == l].sum()), int(l)) for l in np.unique(labs[labs >= 0])]
keep = [l for en, l in sorted(ids, reverse=True) if en >= MIN_E_LABEL][:MAX_LEGEND]
PALETTE = ["#1f77b4", "#d62728", "#2ca02c", "#ff7f0e", "#9467bd", "#8c564b",
           "#e377c2", "#17becf", "#bcbd22", "#7f7f7f", "#aec7e8", "#ff9896",
           "#98df8a", "#ffbb78", "#c5b0d5", "#c49c94", "#f7b6d2", "#9edae5"]
fig = go.Figure()
other = ~np.isin(labs, keep)
fig.add_trace(go.Scatter3d(x=zs[other], y=ys[other], z=xs[other], mode="markers",
                           marker=dict(size=3, color="lightgray", opacity=0.5),
                           name=f"small/<{MIN_E_LABEL:.0f} MeV + noise ({other.sum()} hits)"))
print(f"{'id':>5} {'E(MeV)':>8} {'hits':>6} {'type':>8} {'also in TPCs':>14}")
for i, l in enumerate(keep):
    mk = labs == l
    en = float(e_m[mk].sum())
    typ = "track" if l < S["split_index"] else "blob/sh"
    tpcs_l = sorted(int(t) for t in S["cluster_to_tpcs"].get(int(l), []) if int(t) != TPC)
    print(f"{l:5d} {en:8.1f} {mk.sum():6d} {typ:>8} {str(tpcs_l) if tpcs_l else '-':>14}")
    fig.add_trace(go.Scatter3d(
        x=zs[mk], y=ys[mk], z=xs[mk], mode="markers",
        marker=dict(size=4, color=PALETTE[i % len(PALETTE)], opacity=0.9),
        name=f"cluster {l}  (E={en:.0f} MeV, {mk.sum()} hits, {typ})"))
fig.update_layout(scene=dict(xaxis_title="z", yaxis_title="y", zaxis_title="x"),
                  legend=dict(title="Front-end clusters", itemsizing="constant"),
                  margin=dict(l=0, r=0, b=0, t=40),
                  title=f"FRONT-END CLUSTERING — ev{EV} TPC {TPC} (before any placement)")
fig.show()


   id   E(MeV)   hits     type   also in TPCs
   15    155.6    518    track    [9, 14, 18]
   27    133.9    468    track       [14, 18]
   30    129.7    343    track           [14]
   28    108.0    262    track           [14]
   38     95.8    175    track [17, 24, 25, 34]
  446     95.4    232  blob/sh              -
   19     92.1    238    track   [10, 12, 14]
   40     71.2    164    track              -
   18     58.6    152    track   [10, 12, 14]
  456     45.7    133  blob/sh              -
  420     28.7     69  blob/sh              -
  453     27.4     41  blob/sh              -
  442     24.0     70  blob/sh              -
  431     21.6     53  blob/sh              -
  455     14.7     47  blob/sh              -
  436     10.5     29  blob/sh              -
  415     10.4     27  blob/sh              -
  427      9.7     33  blob/sh           [17]
  439      9.4     19  blob/sh              -
  437      6.6     18  blob/sh              -
  449      5.1     15  blob/sh  

## Stage 2 + 3.1 — Phase 1, then the merged permutation pool

Stage 2 places the most constrained objects first: **showers**, then **tracks
crossing >= 3 TPCs** (at their production t0). Stage 3.1 then places EVERYTHING
big in one pool — the remaining 1-2-TPC backbone tracks **and every P2-class
cluster** (multi-TPC or > 30 MeV, imaged): seed = most energetic unplaced,
batch = shared-TPC mates within the 40% energy band (cap 5), **all orderings
tried** (up to 120 paths), each placing greedily at its scan optimum; the best
ordering by union-support loss commits its first N-1 members. A commit that
fires the overflow trigger is deferred and re-placed later with more context.
Watch for `P2-class cluster` vs `backbone track` in the frame notes.

In [22]:
W.stack2_backbone(S, show_state)


  frame   1 | [2 phase1]  >=3-TPC track label 15 placed at production t0=610.0 (E=435 MeV, truth=609.6031944616698)
Total energy in TPC 16: 1245.94



  frame   2 | [2 phase1]  >=3-TPC track label 18 placed at production t0=68.0 (E=538 MeV, truth=67.83909912337549)
Total energy in TPC 16: 1245.94



  frame   3 | [2 phase1]  >=3-TPC track label 19 placed at production t0=111.0 (E=541 MeV, truth=111.00957239978015)
Total energy in TPC 16: 1245.94



  frame   4 | [2 phase1]  >=3-TPC track label 27 placed at production t0=618.0 (E=421 MeV, truth=617.8578777908115)
Total energy in TPC 16: 1245.94



  frame   5 | [2 phase1]  >=3-TPC track label 38 placed at production t0=350.0 (E=365 MeV, truth=349.81373173650354)
Total energy in TPC 16: 1245.94


...


  frame   8 | [3.1 merged pool]  backbone track 40 (E=71 MeV) COMMITTED at t0=587 (truth=588.214318966493)
Total energy in TPC 16: 1245.94



  frame   9 | [3.1 merged pool]  P2-class cluster 446 (E=95 MeV) DEFERRED by overflow gate at t0=587 (truth=588.111017976189)
Total energy in TPC 16: 1245.94



  frame  10 | [3.1 merged pool]  P2-class cluster 456 (E=46 MeV) COMMITTED at t0=248 (truth=588.2887498300988)
Total energy in TPC 16: 1245.94



  frame  11 | [3.1 merged pool]  P2-class cluster 427 (E=21 MeV) DEFERRED by overflow gate at t0=297 (truth=249.34410277637653)
Total energy in TPC 16: 1245.94



  frame  12 | [3.1 merged pool done]  pool placed 179 objects (90 P2-class, 89 backbone) + swap pass (1 swaps) + rescue pass (35 flagged, 4 moved); deferred once: 37
Total energy in TPC 16: 1245.94


## Stage 3.2 — overflow trigger + saving algorithm

Every placement is probed at its own flash tick against the **full model** (all
stage-2/3.1 light stacked): >= 3 channels with excess >= 500 ADC and z > 2, OR
cos(own, residual) < 0.75. Flagged placements are grouped by slot and resolved:
Stage A anchored permutations, Stage B cross-TPC part detachment (the cut part
becomes a NEW cluster placed in 3.3), N=1 unanchored rescan. Merged P2-class
clusters can fire and be rescued here exactly like backbone tracks.

In [23]:
W.rescue_stage(S, show_state)


  frame  13 | [3.2 trigger]  trigger fired on 47 placements -> 30 co-located groups + 11 lone; running Stage A/B/N=1 decisions
Total energy in TPC 16: 1245.94



  frame  14 | [3.2 saving done]  1 moves, 2 detached parts
Total energy in TPC 16: 1245.94


## Stage 3.3 — Phase-2 scan (what 3.2 detached) + police

(The old stage 3.5 — V2 spatial+light rescue — is **skipped in v0.8.6**: the
joint solver and PASS 2 handle its failure classes better, at zero measured
accuracy cost and −53 s/event. Set `W.SKIP_V2 = False` before `load_chain()`
to replay the historical chain.)


In [24]:
W.staged_stage(S, show_state)

Backbone integrity OK after P2(walkthrough): 157121/157121 decided hits unchanged.

  frame  15 | [3.3 Phase 2]  placed 0 clusters in watched TPC: []
Total energy in TPC 16: 1245.94



  frame  16 | [3.5 V2]  SKIPPED (v0.8.6 canonical: V2 fully superseded by joint + PASS 2 — measured on 126 events)
Total energy in TPC 16: 1245.94


## Stages 3.9 & 4 — joint solver, error matrix, PASS 2 trading, swap fix, force

The v0.8.1 tail of the chain:

- **3.9 JOINT** — the mean-field simultaneous solver owns every TPC ≤ 1.4 GeV:
  all unassigned imaged clusters get placed *together* (light χ² is exactly
  pairwise-quadratic, plus the geodesic-connectivity prior in energy units).
  Every winner touching this TPC is replayed with its q_max and connectivity.
- **4.1 error matrix** — the v11 matrix mops up the template-poor leftovers.
- **3.95 PASS 2** — spatial trading between reco families (v4 score + live
  light veto) with **light-nominated candidates**: families whose t0 the
  cluster's own loss curve finds near-optimal enter the candidate set even if
  spatially distant. Each trade shows whether it was spatial or LIGHT-nominated
  and whether it FIXED or BROKE the cluster against truth.
- **4.5 swap fix**, **4.9 force coverage** — unchanged.


In [25]:
W.smallcluster_stage(S, show_state)


  frame  17 | [3.9 joint]  joint placed cluster 420 (E=28.7 MeV) at t0=482 | q_max 1.00, connected=False (truth=481.7609766469104)
Total energy in TPC 16: 1245.94



  frame  18 | [3.9 joint]  joint placed cluster 453 (E=27.4 MeV) at t0=589 | q_max 1.00, connected=False (truth=249.64829014788847)
Total energy in TPC 16: 1245.94



  frame  19 | [3.9 joint]  joint placed cluster 442 (E=24.0 MeV) at t0=251 | q_max 1.00, connected=False (truth=588.1879432417918)
Total energy in TPC 16: 1245.94



  frame  20 | [3.9 joint]  joint placed cluster 431 (E=21.6 MeV) at t0=589 | q_max 1.00, connected=False (truth=480.8729650103487)
Total energy in TPC 16: 1245.94



  frame  21 | [3.9 joint]  joint placed cluster 455 (E=14.7 MeV) at t0=252 | q_max 1.00, connected=False (truth=588.2540667953435)
Total energy in TPC 16: 1245.94


...


  frame  46 | [3.95 PASS 2 trade]  cluster 454 (E=1.9 MeV) traded 251 -> 588 [spatial] (truth=588.1687729124678, FIX)
Total energy in TPC 16: 1245.94



  frame  47 | [3.95 PASS 2 trade]  cluster 452 (E=1.7 MeV) traded 252 -> 588 [spatial] (truth=588.1891690951306, FIX)
Total energy in TPC 16: 1245.94



  frame  48 | [3.95 PASS 2 done]  trades 91 event-wide (via light nomination 14, vetoed 17; scored 823 of 916 small clusters)
Total energy in TPC 16: 1245.94



  frame  49 | [4.5 swap fix]  0 swaps event-wide (0 MeV)
Total energy in TPC 16: 1245.94



  frame  50 | [4.9 force coverage]  imaged 1 | no-image 347 | noise hits 2959 -> coverage 100%
Total energy in TPC 16: 1245.94


## Final summary — timings for every stage (whole-event work) + this TPC's outcome

In [26]:
import pandas as pd
tdf = pd.DataFrame(S["timings"], columns=["stage", "seconds"])
tdf["seconds"] = tdf["seconds"].round(1)
display(tdf)
print(f"TOTAL {tdf['seconds'].sum():.1f}s for the whole event (all TPCs)")

summ = W.tpc_summary(S)
print(f"\nwatched TPC {TPC}: truth-valid E {summ['e_truth']:.0f} MeV | "
      f"coverage {summ['coverage']*100:.1f}% | efficiency {summ['eff']*100:.2f}%")
show_state("final", "final assignment state", force=True)

,stage,seconds
0,1 clustering & ML prediction (+production plac...,52.1
1,2+3.1 phase1 + merged permutation pool,22.5
2,2+3.1 phase1 + merged permutation pool,17.6
3,3.2 trigger + saving algorithm,11.2
4,3.3a namespace build,0.6
5,3.3 Phase 2 scan (3.2-detached parts + leftovers),1.4
6,3.3b police (post-P2),0.1
7,3.9 joint assignment,7.1
8,4.1 error-matrix Phase 3,22.8
9,3.95 PASS 2 trading (light-nom),6.8


TOTAL 154.6s for the whole event (all TPCs)

watched TPC 16: truth-valid E 1246 MeV | coverage 100.0% | efficiency 82.18%

  frame  51 | [final]  final assignment state
Total energy in TPC 16: 1245.94


In [27]:
# ===== TRUTH-SIDE displays for the watched TPC ==============================
# (1) 3D HTML: hits grouped by TRUE t0 — how the event SHOULD be assembled
# (2) 3D HTML: final reco grouping, for side-by-side rotation
# (3) the classic 2D display (charge center, light both sides) with the
#     prediction rendered AT TRUTH TIMES — "if everything were correct,
#     this is the light we would predict"
import os

OUTDIR = f"{TESTING}/wt_displays"
os.makedirs(OUTDIR, exist_ok=True)
m = S["hit_tpc"] == TPC
xs, ys, zs = (np.asarray(S["fs"].event.x)[m], np.asarray(S["fs"].event.y)[m],
              np.asarray(S["fs"].event.z)[m])

# ---- (1) truth 3D: group by TRUE t0 ----
tru = np.asarray(S["truth_t0"], np.float64)
safe_tru = np.where(np.isfinite(tru) & (tru >= 0), tru, -1.0)
g_tru = pt.group_hits_by_time(safe_tru[m], time_window=10)
p_tru = os.path.abspath(f"{OUTDIR}/ev{EV}_tpc{TPC}_truth3d.html")
pt.plot_3d_clusters_with_t0(
    x=xs, y=ys, z=zs, labels=g_tru, t0s=safe_tru[m], energies=S["e_pos"][m],
    title=f"TRUTH grouping — ev{EV} TPC {TPC}", save_path=p_tru)

# ---- (2) final reco 3D, saved for rotation next to the truth ----
ts_fin = W.current_ts(S)
safe_rec = np.where(np.isfinite(ts_fin) & (ts_fin >= 0), ts_fin, -1.0)
g_rec = pt.group_hits_by_time(safe_rec[m], time_window=10)
p_rec = os.path.abspath(f"{OUTDIR}/ev{EV}_tpc{TPC}_reco3d.html")
pt.plot_3d_clusters_with_t0(
    x=xs, y=ys, z=zs, labels=g_rec, t0s=safe_rec[m], energies=S["e_pos"][m],
    title=f"FINAL reco grouping — ev{EV} TPC {TPC}", save_path=p_rec)

# ---- (3) 2D light+charge with the TRUTH-rendered prediction ----
truth_base = np.zeros_like(S["actual"][TPC], dtype=np.float32)
n_tr, n_no = 0, 0
for (lab, c), img in S["image_maps"].items():
    if int(c) != TPC:
        continue
    tr = W.label_truth(S, int(lab))
    if tr is None or not np.isfinite(tr) or tr < 0:
        n_no += 1
        continue
    truth_base += S["ctx"]["ti"](np.asarray(img, np.float64),
                                 float(tr)).astype(np.float32)
    n_tr += 1
print(f"truth-rendered base: {n_tr} cluster templates at their TRUE t0 "
      f"({n_no} without truth skipped)")
plot_tpc_light_and_charge(ev_id=EV, TPCid=TPC, hits_full_ev=S["hits_full_ev"],
                          actual=S["actual"][TPC].astype(np.float32),
                          predicted=truth_base, geom_map=geom_map,
                          individualScaling=False)
plt.show()

print("interactive 3D (open/rotate):")
print(" ", p_tru)
print(" ", p_rec)


truth-rendered base: 36 cluster templates at their TRUE t0 (0 without truth skipped)
Total energy in TPC 16: 1245.94
interactive 3D (open/rotate):
  /global/cfs/cdirs/dune/users/yuxuan/NDLAr-full/CLMatching_v0.1_backboneOnly/testing/wt_displays/ev1_tpc16_truth3d.html
  /global/cfs/cdirs/dune/users/yuxuan/NDLAr-full/CLMatching_v0.1_backboneOnly/testing/wt_displays/ev1_tpc16_reco3d.html
